In [1]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_groq import ChatGroq


/Users/pratyushsingh/Documents/projects/ai-impl/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = ChatGroq(model="llama-3.1-8b-instant")

In [3]:
parser = JsonOutputParser()

In [13]:
prompt = PromptTemplate(template="provide me with the name,age,city of fictional person in this format {specific_format}",input_variables=[],partial_variables={'specific_format': parser.get_format_instructions()})

In [6]:
prompt

PromptTemplate(input_variables=[], input_types={}, partial_variables={'specific_format': 'Return a JSON object.'}, template='provide me with the name,age,city of fictional person and {specific_format}')

In [14]:
final_prompt = prompt.format()

In [15]:
final_prompt

'provide me with the name,age,city of fictional person in this format Return a JSON object.'

In [16]:
res = model.invoke(final_prompt)

In [17]:
res.content

'Here\'s a JSON object representing a fictional person:\n\n```json\n{\n  "name": "Evelyn Stone",\n  "age": 32,\n  "city": "Portland"\n}\n```\n\nThis represents a person named Evelyn Stone, who is 32 years old and lives in Portland.'

In [23]:
## for getting proper json format we use parser

res_parser = parser.parse(res.content)

In [25]:
res_parser['name']

'Evelyn Stone'

In [ ]:

## we do with chain pipeline as well and getting same result including parsing
chain = prompt | model | parser

result = chain.invoke({})

In [22]:
result['name']

'Evelyn Starling'

### Pydantic model for parsing json output

In [32]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel
from pydantic import BaseModel,Field



In [33]:
class Data(BaseModel):
    name: str=Field(description="name of the person")
    age: int = Field(gt=21,description="age of the person")
    city: str = Field(description="the city has to be of Bihar state")
    state:str = Field(description="the state has to be Bihar only")

In [60]:
parser = PydanticOutputParser(pydantic_object=Data)

In [36]:
parser

PydanticOutputParser(pydantic_object=<class '__main__.Data'>)

In [62]:
prompt = PromptTemplate(
    template="""
Provide name, age, city, state of a fictional {nationality} person.

{format}

Return ONLY a valid JSON object.
Do NOT include explanation, text, or markdown.
""",
    input_variables=['nationality'],
    partial_variables={'format': parser.get_format_instructions()}
)

In [63]:
prompt

PromptTemplate(input_variables=['nationality'], input_types={}, partial_variables={'format': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"name": {"description": "name of the person", "title": "Name", "type": "string"}, "age": {"description": "age of the person", "exclusiveMinimum": 21, "title": "Age", "type": "integer"}, "city": {"description": "the city has to be of Bihar state", "title": "City", "type": "string"}, "state": {"description": "the state has to be Bihar only", "title": "State", "type": "string"}}, "required": ["name", "age", "city", "state"]}\n```'}

In [64]:
final_prompt = prompt.invoke({'nationality': 'Indian'})

In [65]:
res = model.invoke(final_prompt)

In [68]:
res_parse = parser.parse(res.content)

In [70]:
res_parser['name']

'Evelyn Stone'

In [71]:
chain = prompt | model | parser 

In [76]:
res_data = chain.invoke({'nationality':"Indian"})

In [79]:
res_data

Data(name='Rahul Kumar', age=35, city='Patna', state='Bihar')